# Camera–LiDAR Hybrid Sync

Stamp-primary temporal sync + paper-closer residual (IMU warp, optional LSD / dynamic filter, stiff Te).

Operational: `tau_sync = tau_stamp + tau_res`  
Diagnostic: `tau_unconst` (does not override sync).

## 0. Imports

In [ ]:
import importlib
import os
import pickle
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np

_submods = (
    "calib.extract", "calib.imu", "calib.lidar", "calib.camera",
    "calib.geometry", "calib.scoring", "calib.pose_warp", "calib.lines",
    "calib.dynamic_filter", "calib.extrinsic_refine", "calib.hybrid_sync", "calib.viz",
)
import calib  # noqa: F401
for _name in _submods:
    importlib.reload(importlib.import_module(_name))
importlib.reload(sys.modules["calib"])

from calib import (
    CameraPipeline,
    IMUPreintegrator,
    McapTimeWindowExtractor,
    closest_frame_id,
    densify_lidar_scans,
    extract_edges,
    lidar_points_to_pixels,
    make_overlay_image,
    make_temporal_comparison,
    mean_alignment_score,
    plot_distance_histograms,
    project_to_range_image,
    run_hybrid_sync,
    stack_overlays_horizontal,
    tau_only_objective,
)

assert hasattr(calib, "run_hybrid_sync")
print("calib OK — hybrid API")

## 1. Configuration

In [ ]:
MCAP_PATH = "huntington.mcap"
CAMERA_TOPIC = "/cam_sync/cam0/image_raw/compressed"
LIDAR_TOPIC = "/ouster/points"
IMU_TOPIC = "/vectornav/imu_uncompensated"

START_TIME = None
DURATION = None

WINDOW_SIZE = 50
SCAN_STRIDE = 1
MAX_POINTS_PER_SCAN = 2000
SIGMA_PX = 5.0

RESIDUAL_BAND = 0.05
RESIDUAL_SAMPLES = 41
EMA_ALPHA = 0.25
ALIAS_WARN_MS = 80.0

USE_POSE_WARP = True
USE_DYNAMIC_FILTER = True
USE_LSD = False          # set True to rebuild DTs with LSD (slower)
REFINE_EXTRINSICS = True

CACHE_DIR = ".calib_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

K = np.array([
    [1501.9374712879626, 0.0, 566.5690420612353],
    [0.0, 1498.8879775647906, 537.1294320963829],
    [0.0, 0.0, 1.0],
], dtype=float)

T_CL_init = np.array([
    [0.99919851,  0.04002921,  0.00000000,  0.15],
    [0.00000000,  0.00000000, -1.00000000, -0.2815789473684212],
    [-0.04002921, 0.99919851,  0.00000000, -0.13157894736842124],
    [0.0,         0.0,         0.0,          1.0],
], dtype=float)

print(f"warp={USE_POSE_WARP} dyn={USE_DYNAMIC_FILTER} lsd={USE_LSD} Te={REFINE_EXTRINSICS}")
print(f"residual ±{RESIDUAL_BAND*1000:.0f} ms")

## 2. Extract / cache

In [ ]:
_tag = "full" if START_TIME is None and DURATION is None else f"{START_TIME}_{DURATION}"
cache_path = os.path.join(CACHE_DIR, f"v1_window_{_tag}.pkl")

if os.path.exists(cache_path):
    print(f"Loading {cache_path}")
    with open(cache_path, "rb") as f:
        data = pickle.load(f)
    camera_frames = data["camera_frames"]
    lidar_scans = data["lidar_scans"]
    imu_measurements = data["imu_measurements"]
else:
    extractor = McapTimeWindowExtractor(
        mcap_path=MCAP_PATH,
        camera_topic=CAMERA_TOPIC,
        lidar_topic=LIDAR_TOPIC,
        imu_topic=IMU_TOPIC,
    )
    camera_frames, lidar_scans, imu_measurements = extractor.extract(
        start_time=START_TIME, duration=DURATION
    )
    with open(cache_path, "wb") as f:
        pickle.dump(
            {"camera_frames": camera_frames, "lidar_scans": lidar_scans,
             "imu_measurements": imu_measurements},
            f, protocol=4,
        )

print(f"Camera={len(camera_frames)}  LiDAR={len(lidar_scans)}  IMU={len(imu_measurements)}")

## 3. IMU + densify

In [ ]:
imu = IMUPreintegrator(gravity=np.array([0.0, 0.0, 9.81]))
t0 = time.time()
dense_scans = densify_lidar_scans(lidar_scans, imu_measurements, imu_integrator=imu)
print(f"Densified {len(dense_scans)} in {time.time()-t0:.1f}s")

In [ ]:
# Paper Fig. 1(a): line/edge features from a point cloud built by fusing
# adjacent frames via IMU motion compensation ("local mapping", Sec. III-B).
#
# calib.lidar.extract_edges flags a range "jump" against *empty* range-image
# cells too — the raw scan only fills ~85% of the 128x1024 grid, so a large
# chunk of what it returns is a rasterization artifact, not a real object
# boundary. This figure uses a stricter, validity-aware edge test (only
# compares neighboring cells that both have a real return) so what's plotted
# is genuinely sparse line features, matching Fig. 1(a). Viz-only — does not
# touch the calibration pipeline's own edge threshold.
from calib.lidar import apply_transform

EDGE_THRESH_VIZ = 3.0  # meters


def _valid_edges(points, threshold=EDGE_THRESH_VIZ, height=128, width=1024, fov_up=22.5, fov_down=-22.5):
    grid = project_to_range_image(points, height=height, width=width, fov_up=fov_up, fov_down=fov_down)
    ranges = np.linalg.norm(grid, axis=2)
    valid = ranges > 0.001
    diff = np.abs(np.diff(ranges, axis=1))
    diff = np.where(valid[:, :-1] & valid[:, 1:], diff, 0.0)
    diff = np.pad(diff, ((0, 0), (0, 1)), constant_values=0)
    return grid[diff > threshold]


def _scan_time(s):
    return s.get("timestamp_sec", s.get("timestamp"))


t_target = dense_scans[len(dense_scans) // 2]["timestamp"]
mid_idx = min(range(len(lidar_scans)), key=lambda i: abs(_scan_time(lidar_scans[i]) - t_target))
mid_idx = max(1, min(mid_idx, len(lidar_scans) - 2))  # keep a prev/next neighbor in range
prev_scan, curr_scan, next_scan = lidar_scans[mid_idx - 1], lidar_scans[mid_idx], lidar_scans[mid_idx + 1]

raw_edges = _valid_edges(curr_scan["points"])
edges_prev = _valid_edges(prev_scan["points"])
edges_next = _valid_edges(next_scan["points"])

T_p2c = imu.preintegrate(imu_measurements, _scan_time(prev_scan), _scan_time(curr_scan))
T_c2n = imu.preintegrate(imu_measurements, _scan_time(curr_scan), _scan_time(next_scan))
dense_edges = np.vstack([
    apply_transform(edges_prev, T_p2c),
    raw_edges,
    apply_transform(edges_next, np.linalg.inv(T_c2n)),
]).astype(np.float32)


def _front_view(pts, min_range=0.5):
    """Forward-facing (x>0) points seen as a camera-like (horizontal, vertical) view."""
    pts = np.asarray(pts, dtype=float)
    mask = pts[:, 0] > min_range
    pts = pts[mask]
    return -pts[:, 1], pts[:, 2]


xs_d, zs_d = _front_view(dense_edges)
xlim = (xs_d.min(), xs_d.max()) if xs_d.size else (-10, 10)
ylim = (zs_d.min(), zs_d.max()) if zs_d.size else (-5, 5)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, pts, title in zip(
    axes,
    [raw_edges, dense_edges],
    ["single LiDAR scan (no IMU fusion)", "IMU-fused 3-scan edges (dense)"],
):
    xs, zs = _front_view(pts)
    ax.set_facecolor("black")
    ax.scatter(xs, zs, s=1.5, c="lime", marker=".")
    ax.set_title(f"{title}  ({xs.size} pts)")
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("IMU + LiDAR: motion-compensated line-feature density (cf. Fig. 1a)")
plt.tight_layout(); plt.show()

## 4. Camera edges + DT

In [ ]:
# Do NOT pickle full camera DTs — multi-GB files routinely truncate (EOFError).
# camera_frames are already in RAM from the window cache; rebuild is ~15–20s.
pipeline = CameraPipeline(canny_low=50, canny_high=150, sigma_px=SIGMA_PX)
cam_times, frame_ids = [], []
t0 = time.time()
for i, frame in enumerate(camera_frames):
    pipeline.process_frame(i, frame["image"])
    del frame["image"]  # FREE MEMORY
    cam_times.append(frame["timestamp_sec"])
    frame_ids.append(i)
    if (i + 1) % 300 == 0:
        print(f"  {i+1}/{len(camera_frames)}")
cam_times = np.asarray(cam_times, float)
frame_ids = np.asarray(frame_ids, int)
print(f"Camera DT in {time.time()-t0:.1f}s  |  frames={len(frame_ids)}")
# Remove any leftover truncated DT cache
_bad = os.path.join(CACHE_DIR, "v1_camera_dt_full.pkl")
if os.path.exists(_bad):
    os.remove(_bad)
    print(f"Removed stale {_bad}")

In [ ]:
# Paper Fig. 1(b): grayscale image -> Canny edges -> distance transform.
demo_fid = int(frame_ids[len(frame_ids) // 2])
vis_bgr = pipeline.debug_show(demo_fid, max_width=1400, max_height=500)
vis_rgb = cv2.cvtColor(vis_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(14, 5))
plt.imshow(vis_rgb); plt.axis("off")
plt.title(f"Camera edges + DT (frame {demo_fid})  |  image  |  Canny edges  |  distance transform  (cf. Fig. 1b)")
plt.tight_layout(); plt.show()

## 5. Hybrid sync

In [ ]:
print("--- Hybrid sync ---")
t0 = time.time()
stream = run_hybrid_sync(
    pipeline=pipeline,
    lidar_scans=dense_scans,
    cam_times=cam_times,
    frame_ids=frame_ids,
    K=K,
    T_CL_init=T_CL_init,
    window_size=WINDOW_SIZE,
    max_points_per_scan=MAX_POINTS_PER_SCAN,
    scan_stride=SCAN_STRIDE,
    residual_band=RESIDUAL_BAND,
    residual_samples=RESIDUAL_SAMPLES,
    ema_alpha=EMA_ALPHA,
    alias_warn_s=ALIAS_WARN_MS / 1000.0,
    imu=imu,
    imu_measurements=imu_measurements,
    use_pose_warp=USE_POSE_WARP,
    use_dynamic_filter=USE_DYNAMIC_FILTER,
    use_lsd=USE_LSD,
    refine_extrinsics_flag=REFINE_EXTRINSICS,
)
print(f"Done in {time.time()-t0:.1f}s")
best_tau = stream.tau_sync
tau_stamp = stream.tau_stamp
tau_res = stream.tau_res
T_CL = stream.T_CL
print(f"USE: tau_sync={best_tau*1000:+.2f} ms = stamp {tau_stamp*1000:+.2f} + res {tau_res*1000:+.2f}")
print(f"DIAG: unconst={stream.tau_unconst*1000:+.2f} ms alias={stream.alias_suspect}")

## 6. Live plots

In [ ]:
w_idx = [w.window_idx for w in stream.window_results]
score_sync = [w.score_at_sync for w in stream.window_results]
score_stamp = [w.score_at_stamp for w in stream.window_results]
tau_stamp_ms = [w.tau_stamp * 1000 for w in stream.window_results]
tau_sync_ms = [w.tau_sync * 1000 for w in stream.window_results]
tau_res_ms = [w.tau_res * 1000 for w in stream.window_results]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot(w_idx, score_stamp, "-", color="red", label="stamp (baseline)")
axes[0].plot(w_idx, score_sync, "-", color="green", label="sync (ours)")
axes[0].set_title("Score_Evaluation")
axes[0].set_xlabel("Round"); axes[0].set_ylabel("score")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(w_idx, tau_stamp_ms, "-", color="red", label="stamp")
axes[1].plot(w_idx, tau_sync_ms, "-", color="green", label="sync")
axes[1].axhline(best_tau * 1000, color="blue", ls="--", label="final tau")
axes[1].set_title("Tau_Evaluation")
axes[1].set_xlabel("Round"); axes[1].set_ylabel("tau [ms]")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(w_idx, tau_res_ms, "-", color="blue", label="residual")
axes[2].axhline(0, color="black", ls=":", alpha=0.5)
axes[2].set_title("Residual_Evaluation")
axes[2].set_xlabel("Round"); axes[2].set_ylabel("tau_res [ms]")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

fig.suptitle("Hybrid sync live (cf. Fig. 6)")
plt.tight_layout(); plt.show()

## 7. Diagnostic vs operational

In [ ]:
mid = stream.window_results[len(stream.window_results)//2]
scan_indices = np.arange(mid.frame_start, mid.frame_end)

def sweep(lo, hi, n=101):
    taus = np.linspace(lo, hi, n)
    vals = [tau_only_objective(np.array([t]), pipeline, dense_scans, cam_times, frame_ids, K, T_CL_init, scan_indices, MAX_POINTS_PER_SCAN) for t in taus]
    vals = np.asarray(vals)
    return taus, vals, int(np.argmin(vals))

taus_w, vals_w, i_w = sweep(-0.5, 0.5, 101)
taus_b, vals_b, i_b = sweep(tau_stamp - RESIDUAL_BAND, tau_stamp + RESIDUAL_BAND, 101)
print(f"unconst best={taus_w[i_w]*1000:+.1f} ms | in-band best={taus_b[i_b]*1000:+.1f} ms | sync={best_tau*1000:+.1f} ms")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(taus_w*1000, vals_w, color="black")
axes[0].axvline(taus_w[i_w]*1000, color="green", ls="-.", label="unconst")
axes[0].axvline(tau_stamp*1000, color="red", ls=":", label="stamp")
axes[0].axvline(best_tau*1000, color="blue", ls="--", label="sync")
axes[0].axvspan((tau_stamp-RESIDUAL_BAND)*1000, (tau_stamp+RESIDUAL_BAND)*1000, alpha=0.15, color="orange")
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_title("(a) Diagnostic ±500 ms")
axes[0].set_xlabel("tau [ms]"); axes[0].set_ylabel("objective")

axes[1].plot(taus_b*1000, vals_b, color="orange")
axes[1].axvline(best_tau*1000, color="blue", ls="--", label="sync")
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_title("(b) Residual band")
axes[1].set_xlabel("tau [ms]"); axes[1].set_ylabel("objective")

fig.suptitle("Searching-optimization sweep (cf. Fig. 7a)")
plt.tight_layout(); plt.show()

## 8. Distance-colored visual check

In [ ]:
scan = dense_scans[len(dense_scans)//2]
t_L = float(scan["timestamp"])
points_L = scan["points"]

taus_show = [-0.05, float(tau_stamp), float(best_tau)]
labels = ["bad (−50 ms)", "primary stamp", "operational sync"]

# Quantitative stats reused from the distance-colored diagnostic (rgb discarded below).
_, stats = make_temporal_comparison(
    pipeline, cam_times, frame_ids, points_L, t_L, K, T_CL_init,
    taus_show, max_points=8000, saturating_px=15.0,
)

# Paper-style panels (Fig. 1c / Fig. 8): uniform green LiDAR points over the
# grayscale image, no distance colormap.
pts = points_L if points_L.shape[0] <= 8000 else points_L[np.linspace(0, points_L.shape[0]-1, 8000, dtype=int)]
panels = []
for tau, lab in zip(taus_show, labels):
    fid = closest_frame_id(cam_times, frame_ids, t_L + tau)
    uv = lidar_points_to_pixels(pts, T_CL_init, K)
    panel = make_overlay_image(
        pipeline, fid, uv, show_edges=False,
        max_width=520, max_height=520,
        label_text=f"{lab}  (tau={tau*1000:+.1f} ms)",
    )
    panels.append(panel)

combo = stack_overlays_horizontal(panels)
plt.figure(figsize=(15, 5))
plt.imshow(combo); plt.axis("off")
plt.title("Projected LiDAR points on image — paper style (cf. Fig. 1c / Fig. 8)")
plt.tight_layout(); plt.show()

for lab, st in zip(labels, stats):
    print(f"{lab:18s} tau={st['tau']*1000:+6.1f} ms mean|d|={st['mean_d']:.2f} px soft={st['soft_score']:.0f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
plot_distance_histograms(stats, [st["distances"] for st in stats], labels, ax=ax)
plt.tight_layout(); plt.show()